In [ ]:
from data_loader.raw_dataloader import RawDataloader
import os
import polars as pl
from pathlib import Path
from main import load_dotenv
from evaluation.metrics import DatasetEvaluation, MGEvalMetric, MusPyMetrics
from evaluation.parsers import MGEvalAdapter, MusPyAdapter
from data_processing.music_representations.helpers.canonical_loader import (
    canonical_dataset_from_frames,
    filter_canonical_dataset,
    load_canonical_dataset,
)

load_dotenv(Path("../.env"))

from evaluation.parsers import MusPyAdapter, MGEvalAdapter
from evaluation.metrics import MusPyMetrics, MGEvalMetric
import polars as pl

MIN_NOTES = 2

def drop_degenerate(dataset, label):
    keep = (
        dataset.notes.group_by("piece_id")
        .len()
        .filter(pl.col("len") >= MIN_NOTES)
        .get_column("piece_id")
    )
    dropped = dataset.pieces.height - keep.len()
    if dropped:
        print(f"{label}: dropped {dropped} piece(s) with fewer than {MIN_NOTES} notes")
    return filter_canonical_dataset(dataset, keep)

def metrics_table(dataset):
    muspy_metrics = MusPyMetrics().compute(MusPyAdapter().convert(dataset))
    mgeval_metrics = MGEvalMetric().compute(MGEvalAdapter().convert(dataset))
    return muspy_metrics.join(mgeval_metrics, on="piece_id", how="inner")

In [ ]:
dataloader = RawDataloader()
dataloader.prepare_dataset(representation="note_table")
loader = dataloader.loader(load_music=dataloader.load_note_table_music)

canonical = load_canonical_dataset(dataloader.storage.materialize("canonical"))

test_ids = canonical.pieces.filter(pl.col("maestro_split") == "test").get_column("piece_id")
test_canonical = filter_canonical_dataset(canonical, test_ids)
test_metrics = metrics_table(test_canonical)
del test_canonical

train_ids = canonical.pieces.filter(pl.col("maestro_split") == "train").get_column("piece_id")
train_canonical = filter_canonical_dataset(canonical, train_ids)
train_metrics = metrics_table(train_canonical)
del train_canonical

In [ ]:
response = []

comparison = DatasetEvaluation(train_metrics, test_metrics).evaluate()
response.append(pl.DataFrame({"model": "test vs train", "metric": list(comparison), "jsd": list(comparison.values())}).sort("jsd", descending=True))
del train_metrics

In [ ]:
for generated_form in os.scandir("/home/bugslayer/Documents/research/data/MusicGeneration/generated_dataset/"):
    print(generated_form)
    generated_canonical = load_canonical_dataset(Path(generated_form.path))
    print(f"candidate: {generated_canonical.pieces.height} pieces, {generated_canonical.notes.height} notes")

    generated_metrics = metrics_table(generated_canonical)
    del generated_canonical
    comparison = DatasetEvaluation(test_metrics, generated_metrics).evaluate()
    response.append(pl.DataFrame({"model": generated_form.name, "metric": list(comparison), "jsd": list(comparison.values())}).sort("jsd", descending=True))

In [ ]:
response

In [ ]:
comparison_table = (
    pl.concat(response, how="vertical")
    .pivot(
        index="metric",
        on="model",
        values="jsd",
        aggregate_function="first",
    )
    .sort("metric")
)

In [ ]:
comparison_table.write_csv("eval_models.csv")